# microWakeWord — Custom wake word training

Full pipeline: test pronunciations, generate samples, train, export `.tflite`.

## Workarounds applied
- `datasets==2.21.0` (newer versions require torchcodec which breaks audio decoding)
- `piper-tts` needed for piper-sample-generator
- Augmentation data downloaded via soundfile/pydub (bypasses datasets Audio decoder)
- microWakeWord `utils.py` patched: stride assertion replaced with truncation
- `clip_duration_ms` must produce frames divisible by stride (3). Use 1500 or 1800, NOT 2000
- `dinner_party_eval` dataset removed from training config (causes OOM on Colab free tier)

In [8]:
# =============================================================================
# Cell 1: CONFIGURATION — Edit these parameters before running
# =============================================================================
# IMPORTANT: Runtime → Change runtime type → T4 GPU (before running any cell)
# Training works on CPU too but takes ~2-3h instead of ~20min.
# If GPU is unavailable (free tier quota), wait 12-24h for reset or use CPU.
# =============================================================================

import json

CONFIG = {
    'WAKE_WORD_NAME': 'hey serenity',         # output directory and model name
    'CLIP_DURATION_MS': 1500,              # 1500 for single word, 1800 for two words
    'PROBABILITY_CUTOFF': 0.85,            # start low (0.5) to test, raise to reduce false positives

    # TEST_VARIANTS: all candidates to listen in Cell 5. Pick the best in Cell 5b.
   'TEST_VARIANTS': [
    'hey serenity',
    'hey se-ren-i-ty',
    'hey serennity',
    'hey sereniti',
    'hey suhrenity',
    'hey serenitee',
],
    'SAMPLES_PER_COMBO': 300,              # samples per variant+voice combo (total = variants x voices x this)

    # Check available languages/voices/qualities at https://rhasspy.github.io/piper-samples/
    'LANGUAGE': 'en_US',                   # piper language code (es_ES, en_US, de_DE, fr_FR, ...)
    'VOICE_QUALITY': 'medium',             # low, medium, or high (not all qualities available for all languages)

    'NEGATIVE_CLASS_WEIGHT': 20,           # increase to reduce false positives (at cost of sensitivity)
    'TRAINING_STEPS': 50000,               # more steps = longer training, potentially better model
    'EVAL_STEP_INTERVAL': 500,            # how often to evaluate (lower = more checkpoints, more RAM)

    # Evaluates false positive rate against long audio during training. Causes OOM on Colab free tier (T4 GPU).
    # Enable on paid Colab or CPU with enough RAM (needs ~4GB extra).
    'USE_DINNER_PARTY_EVAL': True,
}

with open('/content/config.json', 'w') as f:
    json.dump(CONFIG, f, indent=2)

print(f"Wake word: {CONFIG['WAKE_WORD_NAME']}")
print(f"Clip duration: {CONFIG['CLIP_DURATION_MS']}ms")
print(f"Test variants: {CONFIG['TEST_VARIANTS']}")
print(f"Language: {CONFIG['LANGUAGE']}, quality: {CONFIG['VOICE_QUALITY']}")
print(f"Dinner party eval: {'ON' if CONFIG['USE_DINNER_PARTY_EVAL'] else 'OFF'}")
print('Config saved to /content/config.json')

Wake word: hey serenity
Clip duration: 1800ms
Test variants: ['hey serenity', 'hey se-ren-i-ty', 'hey serennity', 'hey sereniti', 'hey suhrenity', 'hey serenitee']
Language: en_US, quality: medium
Dinner party eval: ON
Config saved to /content/config.json


In [2]:
# Cell 2: Install dependencies
# Safe to re-run — skips what's already installed/cloned.

%pip install 'datasets==2.21.0'
%pip install 'git+https://github.com/whatsnowplaying/audio-metadata@d4ebb238e6a401bb1a5aaaac60c9e2b3cb30929f'
!test -d /content/microWakeWord || git clone https://github.com/kahrendt/microWakeWord /content/microWakeWord
%pip install -e /content/microWakeWord
%pip install piper-tts soundfile pydub

import importlib, site
importlib.invalidate_caches()
site.main()

import datasets
try:
    import microwakeword
    print(f'microwakeword OK')
except ModuleNotFoundError:
    raise RuntimeError('microWakeWord install failed. Scroll up for pip errors.')

if datasets.__version__ != '2.21.0':
    print('\n*** RESTART REQUIRED: old datasets version cached in memory. ***')
    print('*** Runtime → Restart session, then continue from Cell 3. ***')
else:
    print('\nReady. Continue with Cell 3.')

INFO: pip is looking at multiple versions of multiprocess to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.3/527.3 kB 40.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 14.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 177.6/177.6 kB 22.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 146.7/146.7 kB 19.0 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.12.0
    Uninstalling fsspec-2025.12.0:
      Successfully uninstalled fsspec-2025.12.0
  Attempting uninstall: dill
    Found existing installation: dill 0.4.1
    Uninstalling dill-0.4.1:
      Successfully uninstalled dill-0.4.1
  Attempting uninstall: multiprocess
    Found existing installation: multiprocess 0.70.19
    Uninstalling multiprocess-0.70.19:
      Successfully uninstalled multiprocess-0.70.19
  Attempting uninstall: datasets
    

In [3]:
# Cell 3: Patch microWakeWord stride assertion bug
# Verifies session was restarted after Cell 2.

import importlib
try:
    import datasets
    importlib.reload(datasets)
    assert datasets.__version__ == '2.21.0', \
        f'datasets {datasets.__version__} loaded, need 2.21.0. Restart session (Runtime → Restart session), then run Cell 1 and continue from here.'
except ImportError:
    raise RuntimeError('datasets not installed. Run Cell 2 first.')

with open('/content/microWakeWord/microwakeword/utils.py', 'r') as f:
    code = f.read()

old = '            assert spectrogram.shape[0] % stride == 0'
if old in code:
    new = '''            remainder = spectrogram.shape[0] % stride
            if remainder != 0:
                spectrogram = spectrogram[:spectrogram.shape[0] - remainder]'''
    code = code.replace(old, new)
    with open('/content/microWakeWord/microwakeword/utils.py', 'w') as f:
        f.write(code)
    print('Patched')
else:
    print('Already patched')

Patched


In [9]:
# Cell 4: Download piper-sample-generator + voices (auto-detected from LANGUAGE)

import os, json, urllib.request

with open('/content/config.json') as f:
    CONFIG = json.load(f)

if not os.path.exists('./piper-sample-generator'):
    !git clone https://github.com/rhasspy/piper-sample-generator
    !pip install torch torchaudio piper-phonemize-cross==1.2.1

voices_json_path = '/content/voices.json'
if not os.path.exists(voices_json_path):
    urllib.request.urlretrieve(
        'https://huggingface.co/rhasspy/piper-voices/resolve/main/voices.json',
        voices_json_path
    )

with open(voices_json_path) as f:
    all_voices = json.load(f)

VOICES = {}
lang = CONFIG['LANGUAGE']
quality = CONFIG['VOICE_QUALITY']
lang_short = lang.split('_')[0]
for key, info in all_voices.items():
    if not key.startswith(lang):
        continue
    if info.get('quality') != quality:
        continue
    onnx_file = f'{key}.onnx'
    parts = key.replace(f'{lang}-', '').rsplit('-', 1)
    speaker = parts[0]
    url = f'https://huggingface.co/rhasspy/piper-voices/resolve/main/{lang_short}/{lang}/{speaker}/{quality}/{onnx_file}'
    VOICES[key] = url

print(f'Found {len(VOICES)} voices for {lang}/{quality}: {list(VOICES.keys())}')

for name, url in VOICES.items():
    onnx_path = f'/content/{name}.onnx'
    json_path = f'{onnx_path}.json'
    if not os.path.exists(onnx_path):
        !wget -q -O {onnx_path} '{url}'
        !wget -q -O {json_path} '{url}.json'

CONFIG['VOICES'] = VOICES
with open('/content/config.json', 'w') as f:
    json.dump(CONFIG, f, indent=2)

print(f'Voices downloaded. Continue with Cell 5 to test variants.')

Found 18 voices for en_US/medium: ['en_US-amy-medium', 'en_US-arctic-medium', 'en_US-bryce-medium', 'en_US-hfc_female-medium', 'en_US-hfc_male-medium', 'en_US-joe-medium', 'en_US-john-medium', 'en_US-kristin-medium', 'en_US-kusal-medium', 'en_US-l2arctic-medium', 'en_US-lessac-medium', 'en_US-libritts_r-medium', 'en_US-ljspeech-medium', 'en_US-mike-medium', 'en_US-norman-medium', 'en_US-reza_ibrahim-medium', 'en_US-ryan-medium', 'en_US-sam-medium']
Voices downloaded. Continue with Cell 5 to test variants.


In [7]:
len(CONFIG['VOICES']), list(CONFIG['VOICES'].keys())

(18,
 ['en_US-amy-medium',
  'en_US-arctic-medium',
  'en_US-bryce-medium',
  'en_US-hfc_female-medium',
  'en_US-hfc_male-medium',
  'en_US-joe-medium',
  'en_US-john-medium',
  'en_US-kristin-medium',
  'en_US-kusal-medium',
  'en_US-l2arctic-medium',
  'en_US-lessac-medium',
  'en_US-libritts_r-medium',
  'en_US-ljspeech-medium',
  'en_US-mike-medium',
  'en_US-norman-medium',
  'en_US-reza_ibrahim-medium',
  'en_US-ryan-medium',
  'en_US-sam-medium'])

In [10]:
# Cell 5: Test pronunciation variants (listen to all candidates)
# 1. Run this cell to hear each variant
# 2. Listen to the audio samples
# 3. In the next cell (5b), type the numbers of the ones you want to keep and press Enter

from IPython.display import Audio, display, HTML
import os, json

with open('/content/config.json') as f:
    CONFIG = json.load(f)

VOICES = CONFIG['VOICES']
MODEL = f'/content/{list(VOICES.keys())[0]}.onnx'

for i, word in enumerate(CONFIG['TEST_VARIANTS']):
    slug = word.replace(' ', '_')
    out_dir = f'/content/test_{slug}'
    os.makedirs(out_dir, exist_ok=True)

    !cd /content/piper-sample-generator && python -m piper_sample_generator \
        "{word}" \
        --model {MODEL} \
        --max-samples 1 \
        --output-dir {out_dir}

    display(HTML(f'<h3>#{i}: {word}</h3>'))
    display(Audio(f'{out_dir}/0.wav'))

DEBUG:__main__:Loading ['/content/en_US-amy-medium.onnx']
DEBUG:piper.voice:Guessing voice config path: /content/en_US-amy-medium.onnx.json
DEBUG:piper.voice:Using CUDA
/usr/local/lib/python3.13/dist-packages/onnxruntime/capi/onnxruntime_inference_collection.py:197: UserWarning: Specified provider 'CUDAExecutionProvider' is not in available provider names.Available providers: 'AzureExecutionProvider, CPUExecutionProvider'
  warnings.warn(
INFO:__main__:Successfully loaded model(s)
DEBUG:piper.voice:text=hey serenity, phonemes=[['h', 'ˈ', 'e', 'ɪ', ' ', 's', 'ɚ', 'ɹ', 'ˈ', 'ɛ', 'n', 'ᵻ', 'ɾ', 'i']]


DEBUG:__main__:Loading ['/content/en_US-amy-medium.onnx']
DEBUG:piper.voice:Guessing voice config path: /content/en_US-amy-medium.onnx.json
DEBUG:piper.voice:Using CUDA
/usr/local/lib/python3.13/dist-packages/onnxruntime/capi/onnxruntime_inference_collection.py:197: UserWarning: Specified provider 'CUDAExecutionProvider' is not in available provider names.Available providers: 'AzureExecutionProvider, CPUExecutionProvider'
  warnings.warn(
INFO:__main__:Successfully loaded model(s)
DEBUG:piper.voice:text=hey se-ren-i-ty, phonemes=[['h', 'ˈ', 'e', 'ɪ', ' ', 's', 'ˈ', 'i', 'ː', 'ɹ', 'ˈ', 'ɛ', 'n', 'ˈ', 'a', 'ɪ', 't', 'ˈ', 'a', 'ɪ']]


DEBUG:__main__:Loading ['/content/en_US-amy-medium.onnx']
DEBUG:piper.voice:Guessing voice config path: /content/en_US-amy-medium.onnx.json
DEBUG:piper.voice:Using CUDA
/usr/local/lib/python3.13/dist-packages/onnxruntime/capi/onnxruntime_inference_collection.py:197: UserWarning: Specified provider 'CUDAExecutionProvider' is not in available provider names.Available providers: 'AzureExecutionProvider, CPUExecutionProvider'
  warnings.warn(
INFO:__main__:Successfully loaded model(s)
DEBUG:piper.voice:text=hey serennity, phonemes=[['h', 'ˈ', 'e', 'ɪ', ' ', 's', 'ɛ', 'ɹ', 'ˈ', 'ɛ', 'n', 'ᵻ', 'ɾ', 'i']]


DEBUG:__main__:Loading ['/content/en_US-amy-medium.onnx']
DEBUG:piper.voice:Guessing voice config path: /content/en_US-amy-medium.onnx.json
DEBUG:piper.voice:Using CUDA
/usr/local/lib/python3.13/dist-packages/onnxruntime/capi/onnxruntime_inference_collection.py:197: UserWarning: Specified provider 'CUDAExecutionProvider' is not in available provider names.Available providers: 'AzureExecutionProvider, CPUExecutionProvider'
  warnings.warn(
INFO:__main__:Successfully loaded model(s)
DEBUG:piper.voice:text=hey sereniti, phonemes=[['h', 'ˈ', 'e', 'ɪ', ' ', 's', 'ˌ', 'ɛ', 'ɹ', 'ə', 'n', 'ˈ', 'i', 'ː', 'ɾ', 'i']]


DEBUG:__main__:Loading ['/content/en_US-amy-medium.onnx']
DEBUG:piper.voice:Guessing voice config path: /content/en_US-amy-medium.onnx.json
DEBUG:piper.voice:Using CUDA
/usr/local/lib/python3.13/dist-packages/onnxruntime/capi/onnxruntime_inference_collection.py:197: UserWarning: Specified provider 'CUDAExecutionProvider' is not in available provider names.Available providers: 'AzureExecutionProvider, CPUExecutionProvider'
  warnings.warn(
INFO:__main__:Successfully loaded model(s)
DEBUG:piper.voice:text=hey suhrenity, phonemes=[['h', 'ˈ', 'e', 'ɪ', ' ', 's', 'u', 'ː', 'ɹ', 'ˈ', 'ɛ', 'n', 'ᵻ', 'ɾ', 'i']]


DEBUG:__main__:Loading ['/content/en_US-amy-medium.onnx']
DEBUG:piper.voice:Guessing voice config path: /content/en_US-amy-medium.onnx.json
DEBUG:piper.voice:Using CUDA
/usr/local/lib/python3.13/dist-packages/onnxruntime/capi/onnxruntime_inference_collection.py:197: UserWarning: Specified provider 'CUDAExecutionProvider' is not in available provider names.Available providers: 'AzureExecutionProvider, CPUExecutionProvider'
  warnings.warn(
INFO:__main__:Successfully loaded model(s)
DEBUG:piper.voice:text=hey serenitee, phonemes=[['h', 'ˈ', 'e', 'ɪ', ' ', 's', 'ˈ', 'ɛ', 'ɹ', 'ə', 'n', 'ˌ', 'a', 'ɪ', 'ɾ', 'i', 'ː']]


In [11]:
# Cell 5b: Select which variants to keep for training
# 1. Run this cell — it will show the variant list and a text input box
# 2. Type the variant numbers to keep, separated by commas (e.g. 0,1)
# 3. Press Enter

import json

with open('/content/config.json') as f:
    CONFIG = json.load(f)

print('Available variants:')
for i, word in enumerate(CONFIG['TEST_VARIANTS']):
    print(f'  #{i}: {word}')

selection = input('\nEnter numbers to keep (comma-separated): ')
indices = [int(x.strip()) for x in selection.split(',')]
CONFIG['VARIANTS'] = [CONFIG['TEST_VARIANTS'][i] for i in indices]

with open('/content/config.json', 'w') as f:
    json.dump(CONFIG, f, indent=2)

total = len(CONFIG['VARIANTS']) * len(CONFIG['VOICES']) * CONFIG['SAMPLES_PER_COMBO']
print(f"\nSelected: {CONFIG['VARIANTS']}")
print(f'Samples: {len(CONFIG["VARIANTS"])} variants x {len(CONFIG["VOICES"])} voices x {CONFIG["SAMPLES_PER_COMBO"]} = {total} total')

Available variants:
  #0: hey serenity
  #1: hey se-ren-i-ty
  #2: hey serennity
  #3: hey sereniti
  #4: hey suhrenity
  #5: hey serenitee

Enter numbers to keep (comma-separated): 0, 2, 4

Selected: ['hey serenity', 'hey serennity', 'hey suhrenity']
Samples: 3 variants x 18 voices x 300 = 16200 total


In [13]:
import tensorflow as tf
print(tf.config.list_physical_devices('GPU'))

[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [12]:
# Cell 6: Generate training samples

import os, json

with open('/content/config.json') as f:
    CONFIG = json.load(f)

VOICES = CONFIG['VOICES']
os.makedirs('/content/generated_samples', exist_ok=True)

MODELS = [f'/content/{name}.onnx' for name in VOICES.keys()]

sample_idx = 0
for word in CONFIG['VARIANTS']:
    for model in MODELS:
        voice_name = model.split('/')[-1].split('-')[2]
        tmp_dir = f'/content/tmp_gen_{word.replace(" ","_")}_{voice_name}'

        !cd /content/piper-sample-generator && python -m piper_sample_generator \
            "{word}" \
            --model {model} \
            --max-samples {CONFIG['SAMPLES_PER_COMBO']} \
            --batch-size 50 \
            --output-dir {tmp_dir}

        for f in sorted(os.listdir(tmp_dir)):
            if f.endswith('.wav'):
                os.rename(f'{tmp_dir}/{f}', f'/content/generated_samples/{sample_idx}.wav')
                sample_idx += 1

print(f'Total samples: {sample_idx}')

Streaming output truncated to the last 5000 lines.
DEBUG:piper.voice:text=hey suhrenity, phonemes=[['h', 'ˈ', 'e', 'ɪ', ' ', 's', 'u', 'ː', 'ɹ', 'ˈ', 'ɛ', 'n', 'ᵻ', 'ɾ', 'i']]
DEBUG:piper.voice:text=hey suhrenity, phonemes=[['h', 'ˈ', 'e', 'ɪ', ' ', 's', 'u', 'ː', 'ɹ', 'ˈ', 'ɛ', 'n', 'ᵻ', 'ɾ', 'i']]
DEBUG:piper.voice:text=hey suhrenity, phonemes=[['h', 'ˈ', 'e', 'ɪ', ' ', 's', 'u', 'ː', 'ɹ', 'ˈ', 'ɛ', 'n', 'ᵻ', 'ɾ', 'i']]
DEBUG:piper.voice:text=hey suhrenity, phonemes=[['h', 'ˈ', 'e', 'ɪ', ' ', 's', 'u', 'ː', 'ɹ', 'ˈ', 'ɛ', 'n', 'ᵻ', 'ɾ', 'i']]
DEBUG:piper.voice:text=hey suhrenity, phonemes=[['h', 'ˈ', 'e', 'ɪ', ' ', 's', 'u', 'ː', 'ɹ', 'ˈ', 'ɛ', 'n', 'ᵻ', 'ɾ', 'i']]
DEBUG:piper.voice:text=hey suhrenity, phonemes=[['h', 'ˈ', 'e', 'ɪ', ' ', 's', 'u', 'ː', 'ɹ', 'ˈ', 'ɛ', 'n', 'ᵻ', 'ɾ', 'i']]
DEBUG:piper.voice:text=hey suhrenity, phonemes=[['h', 'ˈ', 'e', 'ɪ', ' ', 's', 'u', 'ː', 'ɹ', 'ˈ', 'ɛ', 'n', 'ᵻ', 'ɾ', 'i']]
DEBUG:piper.voice:text=hey suhrenity, phonemes=[['h', 'ˈ', 'e', 'ɪ', ' ', 

In [14]:
# Cell 7: Download augmentation data (MIT RIRs, AudioSet, FMA)

import soundfile as sf
import scipy
import os
import numpy as np
from pathlib import Path
from tqdm import tqdm
from datasets import load_dataset, Audio
import io

# MIT Room Impulse Responses
output_dir = './mit_rirs'
if not os.path.exists(output_dir) or len(os.listdir(output_dir)) == 0:
    os.makedirs(output_dir, exist_ok=True)
    rir_dataset = load_dataset('davidscripka/MIT_environmental_impulse_responses', split='train')
    rir_dataset = rir_dataset.cast_column('audio', Audio(decode=False))
    for i, row in enumerate(tqdm(rir_dataset)):
        audio_path = row['audio']['path']
        if audio_path and os.path.exists(audio_path):
            data, sr = sf.read(audio_path)
            if len(data.shape) > 1:
                data = data[:, 0]
            if sr != 16000:
                import librosa
                data = librosa.resample(data, orig_sr=sr, target_sr=16000)
            scipy.io.wavfile.write(os.path.join(output_dir, f'rir_{i}.wav'), 16000, (data*32767).astype(np.int16))

# AudioSet
# --- replaces the dead AudioSet block (bal_train09.tar is gone; dataset moved to Parquet) ---
import os, numpy as np, soundfile as sf, librosa
from pathlib import Path
from scipy.io import wavfile

if not os.path.exists('esc50'):
    os.mkdir('esc50')
    !wget -q -O esc50/esc50.zip https://github.com/karolpiczak/ESC-50/archive/refs/heads/master.zip
    !cd esc50 && unzip -q esc50.zip

output_dir = './audioset_16k'          # keep the name Cell 8 expects
os.makedirs(output_dir, exist_ok=True)
if not os.listdir(output_dir):
    for wav in Path('esc50').glob('**/audio/*.wav'):
        try:
            data, sr = sf.read(str(wav))
            if data.ndim > 1:
                data = data[:, 0]
            if sr != 16000:
                data = librosa.resample(data, orig_sr=sr, target_sr=16000)
            wavfile.write(os.path.join(output_dir, wav.stem + '.wav'),
                          16000, (data * 32767).astype(np.int16))
        except Exception:
            pass

print(len(os.listdir(output_dir)), 'background clips ready')   # ← expect ~2000, NOT 0

# FMA
if not os.path.exists('fma') or not os.path.exists('fma/fma_small'):
    os.makedirs('fma', exist_ok=True)
    fname = 'fma_xs.zip'
    link = 'https://huggingface.co/datasets/mchl914/fma_xsmall/resolve/main/' + fname
    !wget -O fma/{fname} {link}
    !cd fma && unzip -q {fname}

output_dir = './fma_16k'
if not os.path.exists(output_dir) or len(os.listdir(output_dir)) == 0:
    os.makedirs(output_dir, exist_ok=True)
    from pydub import AudioSegment
    for mp3 in tqdm(list(Path('fma/fma_small').glob('**/*.mp3'))):
        try:
            audio = AudioSegment.from_mp3(str(mp3)).set_frame_rate(16000).set_channels(1)
            samples = np.array(audio.get_array_of_samples(), dtype=np.int16)
            scipy.io.wavfile.write(os.path.join(output_dir, mp3.stem + '.wav'), 16000, samples)
        except Exception:
            pass

print('Augmentation data ready')

Resolving data files:   0%|          | 0/270 [00:00<?, ?it/s]

Generating train split:   0%|          | 0/270 [00:00<?, ? examples/s]

100%|██████████| 270/270 [00:00<00:00, 2035.70it/s]


--2026-10-09 00:18:01--  https://huggingface.co/datasets/agkphysics/AudioSet/resolve/main/data/bal_train09.tar
Resolving huggingface.co (huggingface.co)... 99.86.18.89, 99.86.18.52, 99.86.18.96, ...
Connecting to huggingface.co (huggingface.co)|99.86.18.89|:443... connected.
HTTP request sent, awaiting response... 404 Not Found
2026-10-09 00:18:01 ERROR 404: Not Found.

tar: This does not look like a tar archive
tar: Exiting with failure status due to previous errors


0it [00:00, ?it/s]

--2026-10-09 00:18:01--  https://huggingface.co/datasets/mchl914/fma_xsmall/resolve/main/fma_xs.zip
Resolving huggingface.co (huggingface.co)... 99.86.18.89, 99.86.18.52, 99.86.18.96, ...
Connecting to huggingface.co (huggingface.co)|99.86.18.89|:443... connected.
HTTP request sent, awaiting response... 

302 Found
Location: https://us.gcp.cdn.hf.co/xet-bridge-us/66ca2060627175be3322dcd0/b15b33649259980016c036a0b2a0fee516e0a9acf3f1adcc561e0abd5efde02b?X-Xet-Cas-Uid=public&user_id=public&xip=UAZdd28DQHQ&response-content-disposition=inline%3B+filename*%3DUTF-8%27%27fma_xs.zip%3B+filename%3D%22fma_xs.zip%22%3B&response-content-type=application%2Fzip&Expires=1791508681&Policy=eyJTdGF0ZW1lbnQiOlt7IlJlc291cmNlIjoiaHR0cHM6Ly91cy5nY3AuY2RuLmhmLmNvL3hldC1icmlkZ2UtdXMvNjZjYTIwNjA2MjcxNzViZTMzMjJkY2QwL2IxNWIzMzY0OTI1OTk4MDAxNmMwMzZhMGIyYTBmZWU1MTZlMGE5YWNmM2YxYWRjYzU2MWUwYWJkNWVmZGUwMmJcXD9YLVhldC1DYXMtVWlkPXB1YmxpYyZ1c2VyX2lkPXB1YmxpYyZ4aXA9VUFaZGQyOERRSFEmcmVzcG9uc2UtY29udGVudC1kaXNwb3NpdGlvbj0qJnJlc3BvbnNlLWNvbnRlbnQtdHlwZT0qIiwiQ29uZGl0aW9uIjp7IkRhdGVMZXNzVGhhbiI6eyJFcG9jaFRpbWUiOjE3OTE1MDg2ODF9fX1dfQ__&Signature=MEYCIQDJq8tBescJIJuv0QjFqn36AC0WmOpnGxUOwqVUY%7EeGSwIhAPCmm0uxZ9Daw-74%7E8HjvL7Ipxp%7EwzW-pGHTgz4ZLqud&Key-Pair-Id=01KXEF4KZ1B6FV465MAWR4M21F&Hash-Algorithm=SHA256 [following]
--2026-

/usr/local/lib/python3.13/dist-packages/pydub/utils.py:300: SyntaxWarning: invalid escape sequence '\('
  m = re.match('([su]([0-9]{1,2})p?) \(([0-9]{1,2}) bit\)$', token)
/usr/local/lib/python3.13/dist-packages/pydub/utils.py:301: SyntaxWarning: invalid escape sequence '\('
  m2 = re.match('([su]([0-9]{1,2})p?)( \(default\))?$', token)
/usr/local/lib/python3.13/dist-packages/pydub/utils.py:310: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(flt)p?( \(default\))?$', token):
/usr/local/lib/python3.13/dist-packages/pydub/utils.py:314: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(dbl)p?( \(default\))?$', token):
100%|██████████| 210/210 [01:08<00:00,  3.06it/s]

Augmentation data ready


In [15]:
# Cell 8: Generate spectrogram features

try:
    from microwakeword.audio.augmentation import Augmentation
    from microwakeword.audio.clips import Clips
    from microwakeword.audio.spectrograms import SpectrogramGeneration
except ModuleNotFoundError:
    raise RuntimeError('microWakeWord not installed. Re-run from Cell 2 (VM was reset).')

from mmap_ninja.ragged import RaggedMmap
import os

clips = Clips(
    input_directory='generated_samples',
    file_pattern='*.wav',
    max_clip_duration_s=None,
    remove_silence=False,
    random_split_seed=10,
    split_count=0.1,
)

augmenter = Augmentation(
    augmentation_duration_s=3.2,
    augmentation_probabilities={
        'SevenBandParametricEQ': 0.1,
        'TanhDistortion': 0.1,
        'PitchShift': 0.1,
        'BandStopFilter': 0.1,
        'AddColorNoise': 0.1,
        'AddBackgroundNoise': 0.75,
        'Gain': 1.0,
        'RIR': 0.9,
        # RIR was 0.5
    },
    impulse_paths=['mit_rirs'],
    background_paths=['fma_16k', 'audioset_16k'],
    background_min_snr_db=-5,
    background_max_snr_db=10,
    min_jitter_s=0.195,
    max_jitter_s=0.205,
)

output_dir = 'generated_augmented_features'
os.makedirs(output_dir, exist_ok=True)

for split in ['training', 'validation', 'testing']:
    out_dir = os.path.join(output_dir, split)
    os.makedirs(out_dir, exist_ok=True)

    split_name = {'training': 'train', 'validation': 'validation', 'testing': 'test'}[split]
    repetition = 2 if split == 'training' else 1
    slide = 1 if split == 'testing' else 10

    spectrograms = SpectrogramGeneration(
        clips=clips,
        augmenter=augmenter,
        slide_frames=slide,
        step_ms=10,
    )

    RaggedMmap.from_generator(
        out_dir=os.path.join(out_dir, 'wakeword_mmap'),
        sample_generator=spectrograms.spectrogram_generator(split=split_name, repeat=repetition),
        batch_size=100,
        verbose=True,
    )

print('Spectrogram features ready')

0it [00:00, ?it/s]

0it [00:00, ?it/s]

0it [00:00, ?it/s]

Spectrogram features ready


In [16]:
# Cell 9: Download pre-generated negative datasets

import os

output_dir = './negative_datasets'
if not os.path.exists(output_dir):
    os.mkdir(output_dir)
    link_root = 'https://huggingface.co/datasets/kahrendt/microwakeword/resolve/main/'
    filenames = ['dinner_party.zip', 'dinner_party_eval.zip', 'no_speech.zip', 'speech.zip']
    for fname in filenames:
        link = link_root + fname
        zip_path = f'negative_datasets/{fname}'
        !wget -O {zip_path} {link}
        !unzip -q {zip_path} -d {output_dir}

print('Negative datasets ready')

--2026-10-09 01:28:49--  https://huggingface.co/datasets/kahrendt/microwakeword/resolve/main/dinner_party.zip
Resolving huggingface.co (huggingface.co)... 13.35.202.121, 13.35.202.34, 13.35.202.97, ...
Connecting to huggingface.co (huggingface.co)|13.35.202.121|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://us.gcp.cdn.hf.co/xet-bridge-us/65e327bc1445a768ed343b8c/228d7e72cd5fdc4e6e57da36b88a4c227d34cb8dc44041078b4c4b65dc75848d?xip=UAZdd28DQHQ&user_id=public&response-content-disposition=inline%3B+filename*%3DUTF-8%27%27dinner_party.zip%3B+filename%3D%22dinner_party.zip%22%3B&X-Xet-Cas-Uid=public&response-content-type=application%2Fzip&Expires=1791512929&Policy=eyJTdGF0ZW1lbnQiOlt7IlJlc291cmNlIjoiaHR0cHM6Ly91cy5nY3AuY2RuLmhmLmNvL3hldC1icmlkZ2UtdXMvNjVlMzI3YmMxNDQ1YTc2OGVkMzQzYjhjLzIyOGQ3ZTcyY2Q1ZmRjNGU2ZTU3ZGEzNmI4OGE0YzIyN2QzNGNiOGRjNDQwNDEwNzhiNGM0YjY1ZGM3NTg0OGRcXD94aXA9VUFaZGQyOERRSFEmdXNlcl9pZD1wdWJsaWMmcmVzcG9uc2UtY29udGVudC1kaXNwb3NpdGlvbj0qJ

In [18]:
# Cell 10: Training config

import json, yaml

with open('/content/config.json') as f:
    CONFIG = json.load(f)

config = {}
config['window_step_ms'] = 10
config['train_dir'] = f"trained_models/{CONFIG['WAKE_WORD_NAME']}"

config['features'] = [
    {
        'features_dir': 'generated_augmented_features',
        'sampling_weight': 2.0,
        'penalty_weight': 1.0,
        'truth': True,
        'truncation_strategy': 'truncate_start',
        'type': 'mmap',
    },
    {
        'features_dir': 'negative_datasets/speech',
        'sampling_weight': 10.0,
        'penalty_weight': 1.0,
        'truth': False,
        'truncation_strategy': 'random',
        'type': 'mmap',
    },
    {
        'features_dir': 'negative_datasets/dinner_party',
        'sampling_weight': 10.0,
        'penalty_weight': 1.0,
        'truth': False,
        'truncation_strategy': 'random',
        'type': 'mmap',
    },
    {
        'features_dir': 'negative_datasets/no_speech',
        'sampling_weight': 5.0,
        'penalty_weight': 1.0,
        'truth': False,
        'truncation_strategy': 'random',
        'type': 'mmap',
    },
]

if CONFIG.get('USE_DINNER_PARTY_EVAL'):
    config['features'].append({
        'features_dir': 'negative_datasets/dinner_party_eval',
        'sampling_weight': 1.0,
        'penalty_weight': 1.0,
        'truth': False,
        'truncation_strategy': 'split',
        'type': 'mmap',
    })

config['training_steps'] = [CONFIG['TRAINING_STEPS']]
config['positive_class_weight'] = [1]
config['negative_class_weight'] = [CONFIG['NEGATIVE_CLASS_WEIGHT']]
config['learning_rates'] = [0.001]
config['batch_size'] = 128
config['time_mask_max_size'] = [0]
config['time_mask_count'] = [0]
config['freq_mask_max_size'] = [0]
config['freq_mask_count'] = [0]
config['eval_step_interval'] = CONFIG['EVAL_STEP_INTERVAL']
config['clip_duration_ms'] = CONFIG['CLIP_DURATION_MS']
config['target_minimization'] = 0.9
config['minimization_metric'] = None
config['maximization_metric'] = 'average_viable_recall'

with open('training_parameters.yaml', 'w') as f:
    yaml.dump(config, f)

dp_status = ' + dinner_party_eval' if CONFIG.get('USE_DINNER_PARTY_EVAL') else ''
print(f"Training config saved for \"{CONFIG['WAKE_WORD_NAME']}\" ({CONFIG['CLIP_DURATION_MS']}ms, neg_weight={CONFIG['NEGATIVE_CLASS_WEIGHT']}{dp_status})")

Training config saved for "hey serenity" (1800ms, neg_weight=40 + dinner_party_eval)


In [19]:
# Cell 11: Train model
# ~20 min on T4 GPU. May appear stuck — Colab doesn't print mini-batch results.

!python -m microwakeword.model_train_eval --pooled 1 --max_pool 1 \
--training_config='training_parameters.yaml' \
--train 1 \
--restore_checkpoint 1 \
--test_tf_nonstreaming 0 \
--test_tflite_nonstreaming 0 \
--test_tflite_nonstreaming_quantized 0 \
--test_tflite_streaming 0 \
--test_tflite_streaming_quantized 1 \
--use_weights "best_weights" \
mixednet \
--pointwise_filters "64,64,64,64" \
--repeat_in_block  "1, 1, 1, 1" \
--mixconv_kernel_sizes '[5], [7,11], [9,15], [23]' \
--residual_connection "0,0,0,0" \
--first_conv_filters 32 \
--first_conv_kernel_size 5 \
--stride 3

I0000 00:00:1791510363.084340   49174 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1791510365.865473   49174 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
INFO:absl:Loading and analyzing data sets.
W0000 00:00:1791510383.989556   49174 gpu_bfc_allocator.cc:47] Overriding orig_value setting because the TF_FORCE_GPU_ALLOW_GROWTH environment variable is set. Original config value was false.
Model: "functional"
┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━

In [21]:
list(CONFIG['VOICES'].keys())

['en_US-amy-medium',
 'en_US-arctic-medium',
 'en_US-bryce-medium',
 'en_US-hfc_female-medium',
 'en_US-hfc_male-medium',
 'en_US-joe-medium',
 'en_US-john-medium',
 'en_US-kristin-medium',
 'en_US-kusal-medium',
 'en_US-l2arctic-medium',
 'en_US-lessac-medium',
 'en_US-libritts_r-medium',
 'en_US-ljspeech-medium',
 'en_US-mike-medium',
 'en_US-norman-medium',
 'en_US-reza_ibrahim-medium',
 'en_US-ryan-medium',
 'en_US-sam-medium']

In [20]:
# Cell 12: Download trained model

from google.colab import files
import os

for root, dirs, fnames in os.walk('trained_models'):
    for f in fnames:
        if f.endswith('.tflite'):
            path = os.path.join(root, f)
            print(f'Downloading: {path}')
            files.download(path)

Downloading: trained_models/hey serenity/tflite_stream_state_internal_quant/stream_state_internal_quant.tflite


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>